In [18]:
import matplotlib.pyplot as plt
import pandas_datareader.data as web
import datetime
import pandas as pd
from dbnomics import fetch_series
from utils import fetch_imf_data
from config import country_dict, us_economy_metrics


In [19]:
# Paths
imf_cpi_path = "imf_cpi.csv"
us_economy_path = "us_economy.csv"

In [20]:
# Fetch IMF/CPI - World data
imf_cpi = fetch_imf_data(country_dict=country_dict, metric_suffix="PCPI_IX", frequency="M").dropna(subset=['value'])
# Show 10 samples
imf_cpi.head(10)
# Export data
imf_cpi.to_csv(imf_cpi_path)

In [22]:
# Timeframe covering Trump baseline through the entirety of the Biden administration
start = datetime.datetime(2000, 1, 1)
end = datetime.datetime(2025, 12, 31)

# Define the exact FRED series codes mapped to your portfolio naming convention
try:
    df_list = []
    for name, code in us_economy_metrics.items():
        print(f"-> Fetching {name} ({code})...")
        # Pull data directly from St. Louis Fed servers
        df = web.DataReader(code, 'fred', start, end)
        # 1. Clean the individual metric's index and rename columns
        df = df.reset_index().rename(columns={'DATE': 'Date', code: 'Value'})
        # 2. Handle frequency alignment: Forward-fill quarterly data before stacking
        # This ensures April and May inherit Q1 data before it gets mixed with monthly metrics
        if name in ['Real_GDP', 'Manufacturing_Investment']:
            # Create a continuous monthly date range to map the quarterly data onto
            monthly_range = pd.date_range(start=start, end=end, freq='MS')
            df = df.set_index('Date').reindex(monthly_range).ffill().reset_index().rename(columns={'index': 'Date'})
        # 3. Add the metadata column so we know which metric this row belongs to
        df['Metric_Name'] = name
        # Reorder columns to look clean: Date | Metric_Name | Value
        df = df[['Date', 'Metric_Name', 'Value']]        
        df_list.append(df)
    print("Consolidating and formatting data frequencies...")
    # Concatenate all tables along the Date axis
    bi_dataset = pd.concat(df_list, axis=0, ignore_index=True)
    # Forward-fill (ffill) the quarterly data (GDP & Investment) so monthly rows aren't blank
    # This prevents relationship errors inside Power BI's model
    bi_dataset = bi_dataset.ffill()
    # Reset index to turn the Date from an index into a normal clean column
    bi_dataset = bi_dataset.reset_index().rename(columns={'DATE': 'Date'})
    # Export to local project directory
    bi_dataset.to_csv(us_economy_path, index=False)
    print(f"Success! Master file saved as '{us_economy_path}'")
    print(bi_dataset.tail(10))

except Exception as e:
    print(f"Pipeline Execution Failed: {e}")

-> Fetching CPI_All_Items (CPIAUCSL)...
-> Fetching Unemployment_Rate (UNRATE)...
-> Fetching Total_Nonfarm_Payrolls (PAYEMS)...
-> Fetching Real_GDP (GDPC1)...
-> Fetching Manufacturing_Investment (C307RX1Q020SBEA)...
Consolidating and formatting data frequencies...
Success! Master file saved as 'us_economy.csv'
      index       Date               Metric_Name    Value
1550   1550 2025-03-01  Manufacturing_Investment  145.228
1551   1551 2025-04-01  Manufacturing_Investment  142.245
1552   1552 2025-05-01  Manufacturing_Investment  142.245
1553   1553 2025-06-01  Manufacturing_Investment  142.245
1554   1554 2025-07-01  Manufacturing_Investment  136.654
1555   1555 2025-08-01  Manufacturing_Investment  136.654
1556   1556 2025-09-01  Manufacturing_Investment  136.654
1557   1557 2025-10-01  Manufacturing_Investment  126.551
1558   1558 2025-11-01  Manufacturing_Investment  126.551
1559   1559 2025-12-01  Manufacturing_Investment  126.551
